# Model using topology data

Getting the data in the right formata

In [2]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import joblib
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
from xgboost import XGBClassifier

In [4]:
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    
    # splitting between daily and hourly features
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)       
    xh = win[pos].reshape(len(pos), -1)                        
    
    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    # getting the names for the columns of x
    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    wv_init_cols = [f"{c}_init" for c in wv]
    wv_final_cols = [f"{c}_final" for c in wv]
    
    all_cols = hourly_cols + vol + wv_init_cols + wv_final_cols

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(
        Y.filter(like='result_committed').to_numpy(np.uint8),
        index=Y.index,
        columns=Y.columns[2:]
    )
    
    x = pd.DataFrame(
        np.hstack([xh, xs]).astype(np.float32),
        index=Y.index,
        columns=all_cols
    )
    
    return x, y


class DataLoader:
    # TODO: Change timestamp data from string to datatime object if available in the data source.
    # TODO: move this in the notebooks (or make it simpler/remove it)

    def __init__(self, data_source=None):
        self.data_source = Path(data_source) if data_source else data_dir

    def _csv(self, folder, filename, date_columns=()):
        file_path = self.data_source / folder / filename
        frame = pd.read_csv(file_path)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        file_path = self.data_source / folder / filename
        with file_path.open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv(
            "kernel",
            "Historical_day_ahead_price_2015_2025.csv",
            ("date",),
        )

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv(
            "kernel",
            "Synthetic_water_value_2015_2024.csv",
            ("date",),
        )

    @cached_property
    def commitment_descision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def commitment_decision_data(self):
        return self.commitment_descision_data

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self): #TODO: CHECK THIS ONE LATER
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")



# Class to create the entire dataset in one go
class Training_set(DataLoader):
    '''
    It imports all the dataset, merging them using the the index (that will be the date)
    '''
 
    def __init__(self, data_source=None):
        super().__init__(data_source)
        self.y = None
        self.x = None
 
    def load_copies(self):
        sources = {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }
        return sources
 
 
    def create_x(self):
        '''
        Create the complete X dataset
        
        TODO: feature engineering and selection
                + time split for easier computation (test model on smaller data)
        '''
        
        data = self.load_copies()
        
        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
                
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data
        #   need to repet thier value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')
        
        # creating the two datset
        #   y stored for future use
        self.y = data['uc'].copy()
        data.pop('uc')
        x = data['day_ahead_price']
        data.pop('day_ahead_price')
        for name in data.keys():
                x = x.join(data[name].add_suffix('_' + name), how='left')
                
        
        self.x = x.copy()
    
    
    def split_small_data(self, begin_train:str ='2015-01-01', end_train: str='2018-12-31', begin_test: str='2019-01-12', end_test: str='2019-12-31'):
        '''
        Return X and Y dataset
        
        Input:
            - beginnig_train (str, dafault=2015-01-01 ): strart date training, format 'yyyy-mm-dd'
            - beginnig_test (str, dafault=2018-12-31 ): strart date testing, format 'yyyy-mm-dd'
            - end_train (str, dafault=2019-01-01 ): end date of training, format 'yyyy-mm-dd'
            - end_test (str, dafault=2019-12-31 ): end date of testing, format 'yyyy-mm-dd'
        '''
        
        self.create_x()
        X, Y = self.x.copy(), self.y.copy()
        x_train, y_train = build_cases(X, Y.loc[begin_train:end_train])
        x_test,  y_test  = build_cases(X, Y.loc[begin_test:end_test])
        return x_train, y_train, x_test, y_test

In [5]:
# uploading the data
ts = Training_set(data_source='./Users/lucaag/data')
ts.create_x()

# sorting the data cronologically
X_all = ts.x.sort_index()
Y_all = ts.y.sort_index()

print(f"Range disponibile in Y: {Y_all.index.min()} ---> {Y_all.index.max()} ({len(Y_all)} casi)")

# split 80% train / 20% test
split_idx = int(len(Y_all) * 0.8)
Y_train_raw = Y_all.iloc[:split_idx]
# leaving 7 days buffer (to avoid overlap)
Y_test_raw  = Y_all.iloc[split_idx + 7:]

x_train, y_train = build_cases(X_all, Y_train_raw)
x_test,  y_test  = build_cases(X_all, Y_test_raw)

print(f"Train cases: {x_train.shape[0]} | Test cases: {x_test.shape[0]}")
print(f"Feature originali: {x_train.shape[1]}")

Range disponibile in Y: 2015-01-01 00:00:00+00:00 ---> 2022-12-31 00:00:00+00:00 (2922 casi)
Train cases: 2337 | Test cases: 578
Feature originali: 3243


In [6]:
def add_topology_features(x_df, topology_dict):
    """
    Costruisce feature fisiche ed economiche basate su Tokke_Vinje_topology.yaml:
    1. Grado di riempimento (vol / max_vol) e quota livello acqua (head in m s.l.m.)
    2. Salto idraulico netto (Net Head) per ogni centrale
    3. Net Water Value (WV_up - WV_down) iniziale e finale per ogni centrale
    4. Volumi aggregati per bacini collegati da tunnel (es. Songavatn+Bitdalsvatn, Totak+Vaamarvatn)
    5. Margini di profitto (Prezzo - Net_WV) e statistiche orarie
    """
    new_feats = {}
    reservoirs = topology_dict['model']['reservoir']
    plants = topology_dict['model']['plant']

    # --- 1. FILLING RATIO E LIVELLO ACQUA (HEAD) PER OGNI BACINO ---
    heads = {}
    for res_name, res_info in reservoirs.items():
        col_name = f"{res_name.lower()}_volume"
        if col_name in x_df.columns:
            vol_vals = x_df[col_name].to_numpy()
            max_v = float(res_info['max_vol'])
            
            # Percentuale di riempimento (0.0 -> 1.0+)
            new_feats[f"topo_{res_name.lower()}_fill_ratio"] = vol_vals / max_v
            
            # Interpolazione curva vol_head dallo YAML per avere i metri s.l.m. dell'acqua
            vx = np.array(res_info['vol_head']['x'], dtype=np.float32)
            vy = np.array(res_info['vol_head']['y'], dtype=np.float32)
            head_vals = np.interp(vol_vals, vx, vy)
            heads[res_name.lower()] = head_vals
            new_feats[f"topo_{res_name.lower()}_head_m"] = head_vals

    # --- 2. MAPPATURA TOPOLOGICA DELLE 9 CENTRALI (Upstream -> Downstream) ---
    # Formato: 'nome_centrale': (['bacini_monte'], 'bacino_valle_o_None')
    plant_links = {
        'vesle_kjela': (['kjelavatn'], 'foersvatn'),
        'kjela':       (['foersvatn', 'bordalsvatn'], 'hyljelihyl'), # tunnel Bordalsvatn->Foersvatn
        'songa':       (['songavatn', 'bitdalsvatn'], 'totak'),      # doppio tunnel in Songa
        'vinje':       (['vaamarvatn', 'totak'], 'vinjevatn'),       # tunnel Totak->Vaamarvatn
        'haukeli':     (['vatjern', 'langeidvatn'], 'vinjevatn'),    # fiume da Langeidvatn
        'tokke':       (['vinjevatn'], 'bandak'),
        'byrte':       (['botnedalsvatn'], 'byrtevatn'),
        'lio':         (['byrtevatn'], 'bandak'),                    # via r_Vest_Vassdraget -> Bandak
        'hogga':       (['bandak'], None),                           # scarica fuori dal sistema
    }

    price_cols = [f"day_ahead_price_t{h}" for h in range(168)]
    prices_matrix = x_df[price_cols].to_numpy() # (n_samples, 168)
    price_mean = prices_matrix.mean(axis=1)
    price_max  = prices_matrix.max(axis=1)
    price_min  = prices_matrix.min(axis=1)
    price_std  = prices_matrix.std(axis=1)

    new_feats['price_week_mean'] = price_mean
    new_feats['price_week_max']  = price_max
    new_feats['price_week_min']  = price_min
    new_feats['price_week_std']  = price_std

    # Individuazione suffisso water value usato in build_cases (_wv_final o _wv)
    wv_suffix = "_wv_final" if f"bandak_wv_final" in x_df.columns else "_wv"
    wv_init_suffix = "_wv_init" if f"bandak_wv_init" in x_df.columns else "_wv"

    for p_name, (up_list, down_res) in plant_links.items():
        main_up = up_list[0]
        
        # A. Salto idraulico netto (Net Head = Head_up - max(Head_down, outlet_line))
        yaml_p_name = 'Vesle Kjela' if p_name == 'vesle_kjela' else p_name.capitalize()
        outlet = float(plants[yaml_p_name]['outlet_line'])
        if main_up in heads:
            down_elev = np.maximum(heads[down_res], outlet) if (down_res and down_res in heads) else outlet
            new_feats[f"topo_{p_name}_net_head"] = heads[main_up] - down_elev

        # B. Volume combinato a monte (fondamentale per centrali con tunnel come Songa, Vinje, Kjela)
        up_vols = [x_df[f"{r}_volume"].to_numpy() for r in up_list if f"{r}_volume" in x_df.columns]
        if up_vols:
            new_feats[f"topo_{p_name}_combined_up_vol"] = np.sum(up_vols, axis=0)

        # C. Net Water Value (WV_up - WV_down) e Trend settimanale (WV_final - WV_init)
        wv_up_fin = x_df[f"{main_up}{wv_suffix}"].to_numpy()
        wv_up_ini = x_df[f"{main_up}{wv_init_suffix}"].to_numpy()
        
        wv_dn_fin = x_df[f"{down_res}{wv_suffix}"].to_numpy() if down_res else np.zeros(len(x_df))
        wv_dn_ini = x_df[f"{down_res}{wv_init_suffix}"].to_numpy() if down_res else np.zeros(len(x_df))

        net_wv_fin = wv_up_fin - wv_dn_fin
        net_wv_ini = wv_up_ini - wv_dn_ini

        new_feats[f"topo_{p_name}_net_wv_final"] = net_wv_fin
        new_feats[f"topo_{p_name}_net_wv_init"]  = net_wv_ini
        new_feats[f"topo_{p_name}_wv_trend"]     = wv_up_fin - wv_up_ini

        # D. Spread Economico di Centrale: Prezzo Mercato vs Valore dell'Acqua a Monte
        new_feats[f"topo_{p_name}_spread_mean"] = price_mean - wv_up_fin
        new_feats[f"topo_{p_name}_spread_max"]  = price_max - wv_up_fin
        new_feats[f"topo_{p_name}_net_spread_mean"] = price_mean - net_wv_fin
        
        # Quante ore nella settimana il prezzo supera il Water Value del bacino a monte?
        hours_profitable = (prices_matrix > wv_up_fin[:, None]).sum(axis=1)
        new_feats[f"topo_{p_name}_profitable_hours"] = hours_profitable.astype(np.float32)

        # E. Inflow settimanale totale e medio sui bacini a monte della centrale
        inflow_sums = np.zeros(len(x_df), dtype=np.float32)
        for r in up_list:
            r_inf_cols = [f"{r}_inflow_t{h}" for h in range(168) if f"{r}_inflow_t{h}" in x_df.columns]
            if r_inf_cols:
                inflow_sums += x_df[r_inf_cols].to_numpy().sum(axis=1)
        new_feats[f"topo_{p_name}_up_inflow_week_sum"] = inflow_sums

    topo_df = pd.DataFrame(new_feats, index=x_df.index, dtype=np.float32)
    return pd.concat([x_df, topo_df], axis=1)

In [7]:
# Aggiunge circa 80 feature fisiche/topologiche ad altissimo valore informativo
x_train_topo = add_topology_features(x_train, ts.topology_data)
x_test_topo  = add_topology_features(x_test, ts.topology_data)

print(f"Nuovo numero di colonne con Topologia: {x_train_topo.shape[1]}")

Nuovo numero di colonne con Topologia: 3371


## Random Forest

In [7]:
# 3. Normalizzazione
scaler = StandardScaler()
X_tr_scaled = np.nan_to_num(scaler.fit_transform(x_train_topo.to_numpy(dtype=np.float32)), nan=0.0)
X_te_scaled = np.nan_to_num(scaler.transform(x_test_topo.to_numpy(dtype=np.float32)), nan=0.0)

In [8]:
def extract_positive_probs(proba_list, y_train):
    """Estrae la probabilità della classe 1 (ON) per tutti i 2352 target."""
    n_samples = proba_list[0].shape[0]
    n_targets = len(proba_list)
    probs_2d = np.zeros((n_samples, n_targets), dtype=np.float32)

    for i, p in enumerate(proba_list):
        if p.shape[1] == 2:
            probs_2d[:, i] = p[:, 1]
        else:
            only_class = y_train.iloc[:, i].iloc[0]
            probs_2d[:, i] = float(only_class)
            
    return probs_2d

In [9]:
# 4. Modello senza PCA
# Usiamo max_features=0.10 (circa 330 feature testate per ogni split) così le 
# feature topologiche vengono pescate costantemente ad ogni nodo!
rf_topo_no_pca = RandomForestClassifier(
    n_estimators=120,
    max_depth=14,
    min_samples_split=12,
    min_samples_leaf=5,
    max_features=0.10,         # Testa il 10% delle colonne (~330) invece di sqrt (~57)
    max_samples=0.75,          # Velocizza il calcolo sui 4 core
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nAddestramento Random Forest (NO PCA + Topologia)...")
with joblib.parallel_backend('threading', n_jobs=-1):
    rf_topo_no_pca.fit(X_tr_scaled, y_train.to_numpy())

# 5. Predizione e valutazione Micro ROC-AUC
raw_preds_topo = rf_topo_no_pca.predict_proba(X_te_scaled)
y_pred_topo = extract_positive_probs(raw_preds_topo, y_train)

auc_topo_no_pca = roc_auc_score(y_test.to_numpy(), y_pred_topo, average="micro")
print(f"\n==========================================")
print(f"NO PCA + Topology Test ROC-AUC (micro): {auc_topo_no_pca:.4f}")
print(f"==========================================")


Addestramento Random Forest (NO PCA + Topologia)...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed: 21.7min
[Parallel(n_jobs=-1)]: Done 120 out of 120 | elapsed: 60.0min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.5s
[Parallel(n_jobs=4)]: Done 120 out of 120 | elapsed:    4.3s finished



NO PCA + Topology Test ROC-AUC (micro): 0.8830


## PCA + Random Forest

In [8]:
# 2. Dividiamo le colonne in 3 blocchi logici:
# - Blocco 1: Prezzi orari (168 colonne) -> ridotti con PCA a 30 componenti
# - Blocco 2: Inflow orari (3024 colonne) -> ridotti con PCA a 50 componenti
# - Blocco 3: Volumi, Water Value e Feature Topologiche (~135 colonne) -> TENUTI DIRETTI!
price_cols  = [c for c in x_train_topo.columns if c.startswith('day_ahead_price_t')]
inflow_cols = [c for c in x_train_topo.columns if '_inflow_t' in c]
direct_cols = [c for c in x_train_topo.columns if c not in price_cols + inflow_cols]

print(f"Colonne Prezzi: {len(price_cols)} | Colonne Inflow: {len(inflow_cols)} | Colonne Dirette/Topo: {len(direct_cols)}")

Colonne Prezzi: 168 | Colonne Inflow: 3024 | Colonne Dirette/Topo: 179


In [9]:
# 3. Scaling e PCA separata solo sulle serie orarie lunghe
scaler_p = StandardScaler()
scaler_i = StandardScaler()
scaler_d = StandardScaler()

pca_p = PCA(n_components=30, random_state=42)
pca_i = PCA(n_components=50, random_state=42)

In [10]:
# Fit su train, transform su test
X_tr_p = pca_p.fit_transform(np.nan_to_num(scaler_p.fit_transform(x_train_topo[price_cols]), nan=0.0))
X_te_p = pca_p.transform(np.nan_to_num(scaler_p.transform(x_test_topo[price_cols]), nan=0.0))

X_tr_i = pca_i.fit_transform(np.nan_to_num(scaler_i.fit_transform(x_train_topo[inflow_cols]), nan=0.0))
X_te_i = pca_i.transform(np.nan_to_num(scaler_i.transform(x_test_topo[inflow_cols]), nan=0.0))

X_tr_d = np.nan_to_num(scaler_d.fit_transform(x_train_topo[direct_cols]), nan=0.0)
X_te_d = np.nan_to_num(scaler_d.transform(x_test_topo[direct_cols]), nan=0.0)

# Uniamo i 3 blocchi (~215 feature totali invece di 3300!)
X_train_final = np.hstack([X_tr_p, X_tr_i, X_tr_d]).astype(np.float32)
X_test_final  = np.hstack([X_te_p, X_te_i, X_te_d]).astype(np.float32)

print(f"Shape finale compatta per il modello: {X_train_final.shape}")

Shape finale compatta per il modello: (2337, 259)


In [ ]:
# 4. Random Forest con max_features=0.25 (così a ogni split valuta ~50 feature su 215, 
# trovando sempre i margini di profitto e i salti idraulici delle centrali!)
rf_hybrid = RandomForestClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.25,         # Fondamentale: guarda il 25% delle feature a ogni nodo
    max_samples=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nAddestramento Random Forest (PCA Oraria + Topologia Diretta)...")
with joblib.parallel_backend('threading', n_jobs=-1):
    rf_hybrid.fit(X_train_final, y_train.to_numpy())

# 5. Valutazione
raw_preds_hybrid = rf_hybrid.predict_proba(X_test_final)
y_pred_hybrid = extract_positive_probs(raw_preds_hybrid, y_train)

auc_hybrid = roc_auc_score(y_test.to_numpy(), y_pred_hybrid, average="micro")
print(f"\n==========================================")
print(f"Hybrid (PCA + Topology) Test ROC-AUC (micro): {auc_hybrid:.4f}")
print(f"==========================================")


Addestramento Random Forest (PCA Oraria + Topologia Diretta)...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  5.1min
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 17.5min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done 150 out of 150 | elapsed:    5.2s finished



Hybrid (PCA + Topology) Test ROC-AUC (micro): 0.8957
